# 02. Geospatial Entity Resolution & Legal Name Normalization

Validates Mexican corporate suffix stripping, geohash spatial blocking (precision 6 ≈ 1.2km), and RapidFuzz pairwise matching.

**Philosophy:** Self-contained functions annotated with `# MIGRATES TO:` for modular extraction into `src/mx_osint/resolution/`.

In [ ]:
# Cell 2.1: Load Raw / Synthetic Fixture
from pathlib import Path
import polars as pl

raw_dir = Path("data/raw")
raw_files = list(raw_dir.glob("*.parquet"))
if raw_files:
    source_file = raw_files[0]
    print(f"Loading cached raw data from {source_file}")
    df = pl.read_parquet(source_file)
else:
    fixture_file = Path("fixtures/synthetic_denue_sample.parquet")
    print(f"Loading fixture fallback from {fixture_file}")
    df = pl.read_parquet(fixture_file)

print(f"Total records loaded: {len(df)}")

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/mex_name_normalizer.py::normalize_mexican_legal_name
import re
import unicodedata

LEGAL_SUFFIX_REGEX = re.compile(
    r"\b("
    r"S\.?\s*A\.?\s*P\.?\s*I\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*B\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*DE\s*C\.?\s*V\.?|"
    r"S\.?\s*DE\s*R\.?\s*L\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*C\.?\s*DE\s*R\.?\s*L\.?|"
    r"S\.?\s*C\.?|"
    r"A\.?\s*C\.?|"
    r"S\.?\s*A\.?|"
    r"SAB\s*DE\s*CV"
    r")\b",
    re.IGNORECASE
)
PUNCTUATION_REGEX = re.compile(r"[,.\-'\"]")

def normalize_mexican_legal_name(name: str) -> str:
    """Normalizes trade/legal names by stripping accents, Mexican corporate suffixes, and punctuation."""
    if not name or not isinstance(name, str):
        return ""
    # 1. Uppercase + strip accents
    text = unicodedata.normalize("NFKD", name)
    text = "".join(c for c in text if not unicodedata.combining(c)).upper()
    # 2. Strip legal suffixes
    text = LEGAL_SUFFIX_REGEX.sub(" ", text)
    # 3. Strip punctuation
    text = PUNCTUATION_REGEX.sub(" ", text)
    # 4. Collapse whitespace
    return " ".join(text.split()).strip()

# 10 Inline verification cases
test_cases = [
    ("GRUPO INDUSTRIAL DEL BAJÍO S.A. DE C.V.", "GRUPO INDUSTRIAL DEL BAJIO"),
    ("TECNOLOGIA Y SOFTWARE S.A.P.I. DE C.V.", "TECNOLOGIA Y SOFTWARE"),
    ("LOGISTICA NACIONAL S. DE R.L. DE C.V.", "LOGISTICA NACIONAL"),
    ("SERVICIOS CONTABLES Y FISCALES S.C.", "SERVICIOS CONTABLES Y FISCALES"),
    ("FUNDACION COMUNITARIA A.C.", "FUNDACION COMUNITARIA"),
    ("COMERCIALIZADORA AGRICOLA S.A.", "COMERCIALIZADORA AGRICOLA"),
    ("COOPERATIVA DE TRANSPORTE S.C. DE R.L.", "COOPERATIVA DE TRANSPORTE"),
    ("BANCO MERCANTIL SAB DE CV", "BANCO MERCANTIL"),
    ("CORPORATIVO NACIONAL, S.A. B. DE C.V.", "CORPORATIVO NACIONAL"),
    ("FARMACIAS DE MÉXICO, S.A. DE C.V.", "FARMACIAS DE MEXICO"),
]
for raw, expected in test_cases:
    assert normalize_mexican_legal_name(raw) == expected, f"Mismatch on {raw}"
print("All 10 inline normalizer assertions passed.")

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/spatial_blocker.py::assign_geohash_block
import polars as pl

def encode_geohash(latitude: float, longitude: float, precision: int = 6) -> str:
    """Encodes coordinates into standard geohash string."""
    base32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    lat_interval, lon_interval = [-90.0, 90.0], [-180.0, 180.0]
    geohash, bits, bit, ch, even = [], [16, 8, 4, 2, 1], 0, 0, True
    while len(geohash) < precision:
        if even:
            mid = (lon_interval[0] + lon_interval[1]) / 2.0
            if longitude > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = (lat_interval[0] + lat_interval[1]) / 2.0
            if latitude > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid
        even = not even
        if bit < 4:
            bit += 1
        else:
            geohash.append(base32[ch])
            bit, ch = 0, 0
    return "".join(geohash)

def assign_geohash_block(df: pl.DataFrame, precision: int = 6) -> pl.DataFrame:
    """Assigns geohash spatial block column for localized O(n·k) blocking."""
    hashes = [encode_geohash(lat, lon, precision) for lat, lon in zip(df["lat"].to_list(), df["lon"].to_list())]
    return df.with_columns(pl.Series("geohash", hashes))

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/fuzzy_matcher.py::match_within_blocks
from typing import List, Dict, Any
import polars as pl
from rapidfuzz import fuzz

def match_within_blocks(df: pl.DataFrame, threshold: float = 0.85) -> pl.DataFrame:
    """Finds pairwise duplicates within each spatial geohash block using RapidFuzz WRatio."""
    matches: List[Dict[str, Any]] = []
    # Group rows by geohash block
    blocks = df.partition_by("geohash", as_dict=True)
    for block_key, block_df in blocks.items():
        n = len(block_df)
        if n < 2:
            continue
        names = block_df["norm_name"].to_list()
        ids = block_df["id"].to_list()
        for i in range(n):
            for j in range(i + 1, n):
                score = fuzz.WRatio(names[i], names[j]) / 100.0
                if score >= threshold:
                    matches.append({
                        "id_a": ids[i],
                        "id_b": ids[j],
                        "name_a": names[i],
                        "name_b": names[j],
                        "match_score": round(score, 4),
                        "geohash": block_key[0] if isinstance(block_key, tuple) else block_key
                    })
    if not matches:
        return pl.DataFrame({
            "id_a": pl.Series([], dtype=pl.Utf8),
            "id_b": pl.Series([], dtype=pl.Utf8),
            "name_a": pl.Series([], dtype=pl.Utf8),
            "name_b": pl.Series([], dtype=pl.Utf8),
            "match_score": pl.Series([], dtype=pl.Float64),
            "geohash": pl.Series([], dtype=pl.Utf8)
        })
    return pl.DataFrame(matches)

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/entity_resolver.py::assign_entity_ids
import hashlib
import polars as pl

def assign_entity_ids(df: pl.DataFrame) -> pl.DataFrame:
    """Assigns stable canonical entity_id based on normalized name and spatial hash."""
    entity_ids = []
    for name, gh in zip(df["norm_name"].to_list(), df["geohash"].to_list()):
        key = f"{name}_{gh[:4]}".encode("utf-8")
        entity_ids.append(f"ent_{hashlib.sha1(key).hexdigest()[:12]}")
    
    # Mark duplicates
    is_duplicate = [False] * len(df)
    seen = set()
    canonical_ids = []
    for idx, eid in enumerate(entity_ids):
        if eid in seen:
            is_duplicate[idx] = True
        else:
            seen.add(eid)
        canonical_ids.append(eid)
    
    return df.with_columns([
        pl.Series("entity_id", entity_ids),
        pl.Series("is_duplicate", is_duplicate),
        pl.Series("canonical_entity_id", canonical_ids)
    ])

In [ ]:
# MIGRATES TO: src/mx_osint/resolution/entity_resolver.py::persist_resolved
from pathlib import Path
import polars as pl

def persist_resolved(df: pl.DataFrame, output_path: Path = Path("data/processed/entities_resolved.parquet")) -> Path:
    """Persists deduplicated and resolved entity table."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.write_parquet(output_path)
    total = len(df)
    dups = df.filter(pl.col("is_duplicate") == True).height
    print(f"\n--- Resolution Summary ---")
    print(f"Total input entities: {total}")
    print(f"Duplicates flagged: {dups} ({dups/max(total, 1)*100:.1f}%)")
    print(f"Unique canonical entities: {total - dups}")
    print(f"Saved to {output_path}")
    return output_path

In [ ]:
# Execute pipeline on current dataset
norm_names = [normalize_mexican_legal_name(n) for n in df["nombre_estab"].to_list()]
df_norm = df.with_columns(pl.Series("norm_name", norm_names))
df_blocked = assign_geohash_block(df_norm, precision=6)
matches_df = match_within_blocks(df_blocked, threshold=0.85)
print(f"Pairwise matches found: {len(matches_df)}")
resolved_df = assign_entity_ids(df_blocked)
out_path = persist_resolved(resolved_df)